# Measurement in Z and X bases

The same quantum state gives different statistics depending on the
measurement basis.  Measuring in $Z$ (computational basis) checks
whether the qubit is $|0\rangle$ or $|1\rangle$.  Measuring in $X$
checks whether it is $|+\rangle$ or $|-\rangle$.

To measure in $X$, apply $H$ before the $Z$-measurement.

In [ ]:
from IPython.display import display
import qiskit as qk
import qiskit_aer as qka

In [ ]:
def show(qc, title):
    print(title)
    print(qc.draw())
    sv = qk.quantum_info.Statevector.from_instruction(
        qc.remove_final_measurements(inplace=False)
    )
    for bits, amp in sv.to_dict().items():
        if abs(amp) > 1e-12:
            print(f"  {amp.real:+.3f}{amp.imag:+.3f}j |{bits}>")
    print()


def measure_z(state_prep, shots=1000):
    """Measure in the Z (computational) basis."""
    qc = qk.QuantumCircuit(state_prep.num_qubits, state_prep.num_qubits)
    qc.compose(state_prep, inplace=True)
    qc.measure(range(state_prep.num_qubits), range(state_prep.num_qubits))
    backend = qka.AerSimulator()
    return backend.run(qk.transpile(qc, backend), shots=shots).result().get_counts()


def measure_x(state_prep, shots=1000):
    """Measure in the X basis (H then Z-measure)."""
    qc = qk.QuantumCircuit(state_prep.num_qubits, state_prep.num_qubits)
    qc.compose(state_prep, inplace=True)
    qc.h(range(state_prep.num_qubits))
    qc.measure(range(state_prep.num_qubits), range(state_prep.num_qubits))
    backend = qka.AerSimulator()
    return backend.run(qk.transpile(qc, backend), shots=shots).result().get_counts()


def print_counts(name, counts, shots):
    print(f"  {name}:")
    for state in sorted(counts, key=counts.get, reverse=True):
        print(f"    |{state}>: {counts[state]:>4}/{shots}  ({counts[state]/shots:.1%})")
    print()

## $|0\rangle$: deterministic in $Z$, random in $X$

Measuring $|0\rangle$ in $Z$ always gives 0.  In $X$ it is 50/50.

In [ ]:
prep = qk.QuantumCircuit(1)
show(prep, "|0>")
print_counts("Z-basis", measure_z(prep), 1000)
print_counts("X-basis", measure_x(prep), 1000)

## $|+\rangle$: random in $Z$, deterministic in $X$

$|+\rangle$ is an eigenstate of $X$, so $X$-measurement always
gives $|+\rangle$.  But $Z$-measurement is random.

In [ ]:
prep = qk.QuantumCircuit(1)
prep.h(0)
show(prep, "|+> = H|0>")
print_counts("Z-basis (random)", measure_z(prep), 1000)
print_counts("X-basis (deterministic)", measure_x(prep), 1000)

## $|-\rangle$: the other $X$-eigenstate

$|-\rangle$ is the $-1$ eigenstate of $X$.

In [ ]:
prep = qk.QuantumCircuit(1)
prep.x(0)
prep.h(0)
show(prep, "|-> = HX|0>")
print_counts("Z-basis (random)", measure_z(prep), 1000)
print_counts("X-basis (deterministic)", measure_x(prep), 1000)

## $|1\rangle$: deterministic in $Z$, random in $X$

In [ ]:
prep = qk.QuantumCircuit(1)
prep.x(0)
show(prep, "|1>")
print_counts("Z-basis", measure_z(prep), 1000)
print_counts("X-basis (random)", measure_x(prep), 1000)

## Summary

- A qubit is an eigenstate of a basis if and only if measurement
  in that basis is deterministic.
- $|0\rangle$, $|1\rangle$ are $Z$-eigenstates.
- $|+\rangle$, $|-\rangle$ are $X$-eigenstates.
- Non-eigenstates give random results proportional to $|\alpha|^2$.